# Cw-CONLI v2 Pipeline — Google Colab

Runs the full v2 experiment pipeline on a GPU runtime:
1. Calibrate NLI temperature
2. Precompute scores (dev + test, standard + realistic)
3. Tune thresholds on dev
4. Evaluate on test
5. Analyze + generate figures

**Runtime**: Select **Runtime → Change runtime type → T4 GPU** before running.

**Checkpointing**: Every precompute step appends to CSV. If Colab disconnects, re-run the cell — it resumes automatically.

**Time estimate**: ~3-6 hours for 20K samples on T4 GPU (vs 10+ hours on CPU).

## 0. Setup & Clone

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Define persistent directory on Drive
DRIVE_RESULTS = '/content/drive/MyDrive/Shaun_FYP_results'
!mkdir -p "$DRIVE_RESULTS"

In [ ]:
# Clone repo (or pull latest if already cloned)
import os
if not os.path.exists('/content/Shaun_FYP'):
    !git clone https://github.com/shaunyogeshwaran/Shaun_FYP.git /content/Shaun_FYP
else:
    !cd /content/Shaun_FYP && git pull

os.chdir('/content/Shaun_FYP')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

## 1. Verify GPU Runtime

The engine auto-detects CUDA when available. Just confirm the T4 is visible.

In [ ]:
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## 2. Smoke Test

Run a tiny subset (20 samples) to verify everything works before committing to the full run.

In [ ]:
# Quick smoke test — should finish in ~2 minutes
!python evaluate.py --precompute --split dev --version v2 --limit 20

# Verify output
import pandas as pd
df = pd.read_csv('results/scores_dev_v2.csv')
print(f'\nSmoke test: {len(df)} rows written')
print(df.head())
print(f'\nColumns: {list(df.columns)}')
print(f'NLI methods: {df["nli_method"].value_counts().to_dict()}')

In [ ]:
# Clean up smoke test file so full run starts fresh
import os
smoke_file = 'results/scores_dev_v2.csv'
if os.path.exists(smoke_file):
    os.remove(smoke_file)
    print(f'Removed {smoke_file} — ready for full run')

## 3. Step 1 — Calibrate NLI Temperature

Collects raw 3-class NLI logits on the dev set and fits a single temperature parameter T.

**Note**: Previous experiments found T≈10 (at optimizer boundary), meaning calibration doesn't help for this task. The v2 pipeline runs with `use_calibration=False` by default, but we collect the data for the thesis anyway.

In [ ]:
!python calibrate.py --split dev

In [ ]:
# Inspect calibration result
import json
with open('results/calibration_temperature.json') as f:
    cal = json.load(f)
print(json.dumps(cal, indent=2))

## 4. Step 2 — Precompute Scores

This is the most time-consuming step. Each cell runs independently — if Colab disconnects, re-run the cell and it picks up from the checkpoint.

### 2a. Dev set (standard retrieval)

In [ ]:
!python evaluate.py --precompute --split dev --version v2

### 2b. Test set (standard retrieval)

In [ ]:
!python evaluate.py --precompute --split test --version v2

### 2c. Dev set (realistic shared-index retrieval, QA only)

In [ ]:
!python evaluate.py --precompute --split dev --realistic --version v2

### 2d. Test set (realistic shared-index retrieval, QA only)

In [ ]:
!python evaluate.py --precompute --split test --realistic --version v2

### Checkpoint: Save precomputed scores to Drive

In [ ]:
# Save CSVs to Drive after precomputation
!cp results/scores_*_v2.csv "$DRIVE_RESULTS/" 2>/dev/null
!cp results/scores_*_realistic_v2.csv "$DRIVE_RESULTS/" 2>/dev/null
!cp results/calibration_*.csv "$DRIVE_RESULTS/" 2>/dev/null
!cp results/calibration_*.json "$DRIVE_RESULTS/" 2>/dev/null
!ls -lh "$DRIVE_RESULTS/"
print('Scores backed up to Google Drive')

## 5. Step 3 — Tune Thresholds

Grid search over C2 and C3 thresholds using dev set scores. This is fast (no model inference).

In [ ]:
# Standard retrieval tuning
!python tune.py --split dev --version v2

In [ ]:
# Realistic retrieval tuning (QA only)
!python tune.py --split dev --version v2 --realistic

In [ ]:
# Per-task tuning (optional, for thesis breakdown)
!python tune.py --split dev --version v2 --task qa
!python tune.py --split dev --version v2 --task summarization

In [ ]:
# Inspect tuning results
import json
with open('results/tuning_results_v2.json') as f:
    tuning = json.load(f)
for name, info in tuning.items():
    print(f"{name:15s}  F1={info['best_f1']:.4f}  {info['best_params']}")

## 6. Step 4 — Evaluate on Test Set

Apply tuned thresholds to the held-out test set.

In [ ]:
# Standard evaluation — all conditions
!python evaluate.py --condition C1 --split test --version v2
!python evaluate.py --condition C2 --split test --version v2
!python evaluate.py --condition C3 --split test --version v2

In [ ]:
# Realistic evaluation
!python evaluate.py --condition C1 --split test --version v2 --realistic
!python evaluate.py --condition C2 --split test --version v2 --realistic
!python evaluate.py --condition C3 --split test --version v2 --realistic

In [ ]:
# Ablation: evaluate with whole-response NLI scores (v2 without decomposition)
!python evaluate.py --condition C2 --split test --version v2 --nli-key nli_score_whole
!python evaluate.py --condition C3 --split test --version v2 --nli-key nli_score_whole

## 7. Step 5 — Analyze & Generate Figures

In [ ]:
# Main analysis: all tasks, test set
!python analyze.py --split test --version v2

In [ ]:
# Per-task analysis
!python analyze.py --split test --version v2 --task qa
!python analyze.py --split test --version v2 --task summarization

In [ ]:
# Realistic retrieval analysis
!python analyze.py --split test --version v2 --realistic

In [ ]:
# Display generated figures inline
from IPython.display import Image, display
import glob

figures = sorted(glob.glob('results/figures/*_v2*.png'))
print(f'Generated {len(figures)} figures:\n')
for fig_path in figures:
    print(f'--- {os.path.basename(fig_path)} ---')
    display(Image(filename=fig_path, width=700))
    print()

## 8. Save Results to Drive & Download

In [ ]:
# Copy all results to Google Drive
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/Shaun_FYP_results'

# Copy entire results directory
if os.path.exists(DRIVE_RESULTS):
    shutil.rmtree(DRIVE_RESULTS)
shutil.copytree('results', DRIVE_RESULTS)

# List saved files
for root, dirs, files in os.walk(DRIVE_RESULTS):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, DRIVE_RESULTS)
        print(f'  {rel:50s} {size:8.1f} KB')

print(f'\nAll results saved to Google Drive: {DRIVE_RESULTS}')

In [ ]:
# Create downloadable zip
shutil.make_archive('/content/results_v2', 'zip', '.', 'results')

from google.colab import files
files.download('/content/results_v2.zip')
print('Download started — check your browser downloads.')

## 9. Quick Summary

In [ ]:
# Print a quick summary of all results
import json, glob

print('=' * 70)
print('v2 EXPERIMENT SUMMARY')
print('=' * 70)

# Tuning results
for tf in sorted(glob.glob('results/tuning_results*v2*.json')):
    print(f'\n--- {os.path.basename(tf)} ---')
    with open(tf) as f:
        data = json.load(f)
    for name, info in data.items():
        print(f'  {name:15s}  F1={info["best_f1"]:.4f}')

# Eval results
print('\n--- Test Set Evaluation ---')
for ef in sorted(glob.glob('results/eval_*_test_*v2*.json')):
    with open(ef) as f:
        data = json.load(f)
    m = data['metrics']
    print(f'  {os.path.basename(ef):45s}  F1={m["f1"]:.4f}  Prec={m["precision"]:.4f}  Rec={m["recall"]:.4f}')

# McNemar results
print('\n--- McNemar Tests ---')
for mf in sorted(glob.glob('results/mcnemar*v2*.json')):
    with open(mf) as f:
        data = json.load(f)
    sig = 'YES' if data['significant'] else 'no'
    print(f'  {os.path.basename(mf):45s}  p={data["p_value"]:.2e}  significant={sig}')

print('\n' + '=' * 70)

---

### Troubleshooting

| Problem | Solution |
|---------|----------|
| `CUDA out of memory` | Restart runtime and re-run from the disconnected cell (checkpointing handles it) |
| `scores_dev_v2.csv not found` | Run the precompute cell for that split first |
| `tuning_results_v2.json not found` | Run the tuning cell first |
| Colab disconnected mid-run | Re-run the interrupted cell — it resumes from checkpoint |
| Drive not mounted | Re-run cell 0 (Setup & Clone) |
| `ModuleNotFoundError` | Re-run the pip install cell |